# 🔍 Forensic AI — Google Colab Setup

**Multi-agent forensic accounting platform**  
Runs entirely in Colab with a **free API key** (Groq or Gemini).

---

## Quick Start
1. Run **Cell 1** (Install packages) — takes ~2 minutes
2. Run **Cell 2** (Mount Drive + set API key)
3. Run **Cell 3** (Import platform)
4. Run **Cell 4** (Investigate a company)

### Free API Options
| Provider | Free Tier | Sign Up |
|----------|-----------|--------|
| **Groq** | 14,400 req/day, fast Llama 70B | [console.groq.com](https://console.groq.com) |
| **Gemini** | 1,500 req/day, Gemini 1.5 Pro | [aistudio.google.com](https://aistudio.google.com/app/apikey) |


In [ ]:
# ── Cell 1: Install Dependencies ─────────────────────────────────
# Uses minimal requirements — cloud API only, no local GPU inference needed.

import subprocess, sys

PACKAGES = [
    # LLM providers
    'openai>=1.35.0',               # Groq + OpenAI + OpenRouter (all OpenAI-compatible)
    'google-generativeai>=0.7.0',   # Google Gemini
    'anthropic>=0.31.0',            # Claude (optional)
    # Core utilities
    'python-dotenv>=1.0.0',
    'loguru>=0.7.2',
    'httpx>=0.27.0',
    'requests>=2.31.0',
    'tenacity>=8.3.0',
    'tqdm>=4.66.0',
    # Financial data
    'yfinance>=0.2.40',
    # Data processing
    'pandas>=2.1.0',
    'numpy>=1.26.0',
    'lxml>=5.1.0',
    'beautifulsoup4>=4.12.3',
    # Fuzzy matching (used in company lookup + helpers)
    'fuzzywuzzy>=0.18.0',
    'python-Levenshtein>=0.25.0',
    # RAG / embeddings
    'sentence-transformers>=3.0.0',
    'chromadb>=0.5.0',
    'rank-bm25>=0.2.2',
    # Document processing
    'PyMuPDF>=1.24.0',              # imported as fitz
    'pdfplumber>=0.11.0',
    'Pillow>=10.3.0',
    # Report generation
    'python-docx>=1.1.0',
    'openpyxl>=3.1.2',
    'python-pptx>=0.6.23',
    'reportlab>=4.2.0',
    # Database
    'duckdb>=1.0.0',
]

print('Installing packages...')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet'] + PACKAGES)
print('✓ All packages installed.')

In [ ]:
# ── Cell 2: Mount Google Drive + Configure API Key ────────────────

import os
from google.colab import drive, userdata

# Mount Google Drive (reports will be saved here)
drive.mount('/content/drive')
DRIVE_OUTPUT = '/content/drive/MyDrive/Forensic_Reports'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print(f'✓ Drive mounted. Reports → {DRIVE_OUTPUT}')

# ── API Key Setup ─────────────────────────────────────────────────
# Store your key in Colab Secrets (🔑 icon in left sidebar) then load it:
#
#   Secret name: GROQ_API_KEY   (recommended — free, fast)
#   Secret name: GOOGLE_API_KEY (alternative — Gemini)
#
# The platform auto-detects whichever key is present.

def _load_secret(name: str) -> str:
    try:
        return userdata.get(name) or ''
    except Exception:
        return ''

groq_key    = _load_secret('GROQ_API_KEY')
gemini_key  = _load_secret('GOOGLE_API_KEY')
openai_key  = _load_secret('OPENAI_API_KEY')
claude_key  = _load_secret('ANTHROPIC_API_KEY')

if groq_key:   os.environ['GROQ_API_KEY']      = groq_key
if gemini_key: os.environ['GOOGLE_API_KEY']    = gemini_key
if openai_key: os.environ['OPENAI_API_KEY']    = openai_key
if claude_key: os.environ['ANTHROPIC_API_KEY'] = claude_key

os.environ['REPORTS_DIR'] = DRIVE_OUTPUT

detected = [n for n, k in [('Groq', groq_key), ('Gemini', gemini_key),
                             ('OpenAI', openai_key), ('Claude', claude_key)] if k]
if detected:
    print(f'✓ API keys detected: {detected}')
    print('  Platform will use:', detected[0], '(highest priority)')
else:
    print('⚠ No API keys found!')
    print('  Add a key in Colab Secrets (🔑 icon) with name GROQ_API_KEY or GOOGLE_API_KEY')
    print('  Groq: https://console.groq.com  |  Gemini: https://aistudio.google.com/app/apikey')

In [ ]:
# ── Cell 3: Clone the Forensic AI repo from GitHub ───────────────

import sys, os, subprocess

REPO_URL = 'https://github.com/anubhav0499-bit/forensic-ai.git'
REPO_DIR = '/content/forensic-ai'

# Clone only if not already present (safe to re-run)
if not os.path.isdir(REPO_DIR):
    print(f'Cloning {REPO_URL} ...')
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR])
    print(f'✓ Cloned to {REPO_DIR}')
else:
    print(f'✓ Repo already present at {REPO_DIR}')
    subprocess.call(['git', '-C', REPO_DIR, 'pull', '--quiet'])

# Add repo root to Python path so `from agents.orchestrator import ...` works
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Verify the platform loads
try:
    from llm.llm_manager import LLMManager
    llm = LLMManager()
    info = llm.get_backend_info()
    print(f'✓ LLM Backend: {info["backend"]}')
    print(f'  Primary model : {info["primary_model"]}')
    print(f'  Fast model    : {info["fast_model"]}')

    if info['backend'] == 'template':
        print()
        print('⚠  No LLM connected. Set an API key in Cell 2 and re-run.')
    else:
        test = llm.fast_generate('Say: FORENSIC AI READY', max_tokens=20)
        print(f'  Test response : {test[:60]}')
        print('✓ Platform ready.')
except ImportError as e:
    print(f'✗ Import error: {e}')
    print(f'  Check that {REPO_DIR} contains agents/, llm/, rag/ folders.')
    print(f'  Try: !ls {REPO_DIR}')

In [ ]:
# ── Cell 4: Run a Forensic Investigation ─────────────────────────

from agents.orchestrator import ForensicOrchestrator

# ← CHANGE THIS to any company name or ticker:
COMPANY = 'RAJESH EXPORTS'
TICKER  = 'RAJESHEXPO'    # optional: helps data acquisition

print(f'Starting forensic investigation: {COMPANY} ({TICKER})')
print('This typically takes 3–8 minutes depending on the LLM backend...')
print()

orchestrator = ForensicOrchestrator()
result = orchestrator.investigate(COMPANY, ticker=TICKER)

# ── Print Summary ─────────────────────────────────────────────────
print()
print('=' * 60)
print(f'FORENSIC INVESTIGATION COMPLETE: {result["company_name"]}')
print('=' * 60)
print(f'Overall Risk Score : {result["overall_risk_score"]:.1f} / 100')
print(f'Verdict            : {result["verdict"]}')
print(f'Red Flags          : {result["red_flags"]}')
print(f'Green Flags        : {result["green_flags"]}')
print(f'CV Issues          : {result["cross_validation_issues"]} ({result["critical_cv_issues"]} CRITICAL)')
print(f'Duration           : {result["investigation_duration_seconds"]}s')
print()

if result.get('top_red_flags'):
    print('TOP RED FLAGS:')
    for flag in result['top_red_flags'][:5]:
        print(f"  [{flag['risk_level']}] {flag['title']}")
    print()

print('Risk Components:')
for dim, score in result.get('risk_components', {}).items():
    bar = '█' * int(score / 10) + '░' * (10 - int(score / 10))
    print(f'  {dim:30s} {bar} {score:.1f}')
print()

print('Reports saved to:')
for fmt, path in result.get('report_paths', {}).items():
    print(f'  {fmt.upper():6s}: {path}')

In [ ]:
# ── Cell 5: LLM Provider Switcher ────────────────────────────────
# Use this cell to switch providers mid-session or test connectivity.

from llm.llm_manager import LLMManager

# Show current backend
llm = LLMManager()
print('Current LLM backend:')
for k, v in llm.get_backend_info().items():
    print(f'  {k}: {v}')

# To force a different provider, set env var before creating LLMManager:
# import os
# os.environ['LLM_PROVIDER'] = 'gemini'   # or groq / openai / anthropic
# llm2 = LLMManager()
# print(llm2.get_backend_info())

# Test the current LLM with a forensic prompt:
test_prompt = (
    'A company shows 40% revenue growth but CFO/NI ratio drops from 1.1x to 0.4x. '
    'What are the top 3 forensic red flags? Answer in 3 bullet points.'
)
print('\nTest prompt response:')
print(llm.fast_generate(test_prompt, max_tokens=300))

## Tips & Troubleshooting

### LLM Not Connecting?
- **Groq**: Verify your key at [console.groq.com](https://console.groq.com/keys) → should start with `gsk_`
- **Gemini**: Get key at [aistudio.google.com](https://aistudio.google.com/app/apikey) → should start with `AIza`
- After setting keys in Colab Secrets (🔑), re-run Cell 2 then Cell 3.

### Changing the LLM Model
```python
import os
# Use Groq's smaller/faster model for quicker results:
os.environ['GROQ_MODEL'] = 'llama-3.1-8b-instant'
# Use Gemini 2.0 Flash for cheapest/fastest:
os.environ['GEMINI_MODEL'] = 'gemini-2.0-flash'
```

### Force a Specific Provider
```python
import os
os.environ['LLM_PROVIDER'] = 'gemini'  # groq | openai | anthropic | gemini
```

### Reports Not Saving to Drive?
Make sure Cell 2 ran successfully (Drive is mounted). Reports save to:  
`/content/drive/MyDrive/Forensic_Reports/<Company>/`

### Speed Guide
| Backend | Time per investigation | Cost |
|---------|----------------------|------|
| Groq (Llama 70B) | ~3–5 min | Free |
| Gemini 1.5 Pro | ~4–6 min | Free tier |
| Gemini 2.0 Flash | ~2–3 min | Free tier |
| GPT-4o | ~5–8 min | ~$0.10 |
| Claude Opus | ~6–10 min | ~$0.15 |
